# 자동차임베디드AI 실습 1주차 공부용 정리

이 노트북은 `lab-01.ipynb` 실습을 다시 공부하기 위한 핵심 정리본이다.

핵심 흐름:

`CSV 로드 -> 타겟 생성 -> 타입 정리 -> Train/Val 분리 -> 누수 방지 전처리 -> SMOTE -> kNN 튜닝 -> PCA 시각화 -> Test 예측 -> CSV 저장 -> NumPy로 kNN 재구현`

공부 포인트:

- 각 단계에서 **어떤 함수 문법**을 쓰는지
- 코드가 **어떤 순서와 흐름**으로 이어지는지
- 그 함수가 **왜 필요한지**를 이해하는 것


## 1. 라이브러리 프레임워크

핵심 역할:

- `pandas`: CSV 로드, 표 데이터 다루기
- `numpy`: 배열 연산, 거리 계산, 브로드캐스팅
- `scikit-learn`: 전처리, 분할, 모델 학습, 평가, PCA
- `imblearn`: SMOTE로 클래스 불균형 보정

원리:

- 전체 파이프라인은 보통 `DataFrame -> numpy array -> model input` 흐름으로 간다.
- 실무에서는 데이터를 읽는 도구와 모델 학습 도구가 분리되어 있기 때문에, 각 라이브러리의 역할을 같이 기억하는 것이 중요하다.


In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, ConfusionMatrixDisplay
from sklearn.decomposition import PCA
from imblearn.over_sampling import SMOTE


## 2. 데이터 로드와 타겟 생성

핵심 문법:

- `pd.read_csv(path)`: CSV를 `DataFrame`으로 읽는다.
- `df['col']`: 특정 열을 선택한다.
- `Series.apply(func)`: 각 원소에 함수를 적용한다.
- `lambda x: ...`: 짧은 익명 함수다.

원리:

- 원래 데이터의 `transmission`은 문자열이다.
- 모델은 문자열 자체를 바로 예측하기보다, `Auto / Manual`을 `0 / 1`로 바꾼 타겟을 학습한다.

흐름:

- 원본 CSV 로드
- `transmission` 열 확인
- 규칙 기반으로 `target` 열 생성


In [ ]:
trainval_df = pd.read_csv('trainval.csv')
test_df = pd.read_csv('test.csv')

trainval_df['target'] = trainval_df['transmission'].apply(
    lambda x: 1 if 'Manual' in str(x) else 0
)


## 3. 타입 정리와 Train/Val 분리

핵심 문법:

- `pd.to_numeric(series, errors='coerce')`
- `astype(str)`
- `train_test_split(X, y, test_size=0.2)`

원리:

- 전처리 전에 먼저 각 열의 타입을 확정해야 한다.
- 수치형은 숫자로 강제 변환하고, 실패한 값은 `NaN`으로 바꿔 나중에 결측치 처리한다.
- 범주형은 문자열로 통일해야 인코더가 안정적으로 동작한다.

중요 포인트:

- 데이터를 먼저 `Train`과 `Val`로 나누는 이유는 **Data Leakage**를 막기 위해서다.
- 이후 기준 학습은 반드시 `Train`으로만 해야 한다.


In [ ]:
numeric_cols = [
    'year',
    'engine_cylinders',
    'engine_displacement',
    'city_mpg_ft1',
    'highway_mpg_ft1',
]

categorical_cols = [
    'make',
    'class',
    'drive',
]

for col in numeric_cols:
    trainval_df[col] = pd.to_numeric(trainval_df[col], errors='coerce')
    test_df[col] = pd.to_numeric(test_df[col], errors='coerce')

for col in categorical_cols:
    trainval_df[col] = trainval_df[col].astype(str)
    test_df[col] = test_df[col].astype(str)

feature_cols = numeric_cols + categorical_cols

X_train_full, X_val_full, y_train, y_val = train_test_split(
    trainval_df[feature_cols],
    trainval_df['target'],
    test_size=0.2,
    random_state=42,
)


## 4. 전처리 파이프라인 문법

핵심 문법:

- `Pipeline(steps=[...])`
- `SimpleImputer(strategy='median')`
- `SimpleImputer(strategy='most_frequent')`
- `StandardScaler()`
- `OneHotEncoder(handle_unknown='ignore', sparse_output=False)`
- `ColumnTransformer(transformers=[...])`

원리:

- 수치형과 범주형은 처리 방식이 다르기 때문에 열 그룹별 체인을 따로 만든다.
- `Pipeline`은 같은 종류의 처리 순서를 묶는 도구다.
- `ColumnTransformer`는 여러 파이프라인을 열 그룹별로 병렬 적용하는 도구다.

흐름:

- 수치형: 결측치 대체 -> 스케일링
- 범주형: 결측치 대체 -> 원-핫 인코딩
- 마지막에 두 결과를 하나의 입력 행렬로 합친다.


In [ ]:
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_cols),
        ('cat', categorical_transformer, categorical_cols),
    ]
)


## 5. `fit`과 `transform`

핵심 문법:

- `preprocessor.fit(X_train_full)`
- `preprocessor.transform(X_val_full)`

원리:

- `fit`: 기준을 학습한다.
- `transform`: 이미 학습한 기준을 적용한다.

왜 중요한가:

- 중앙값, 평균, 분산, 범주 목록 같은 기준을 `Val/Test`까지 보고 만들면 미래 정보를 미리 본 것과 같아진다.
- 그래서 `Train`으로만 `fit`하고, `Val/Test`에는 `transform`만 해야 한다.


In [ ]:
preprocessor.fit(X_train_full)

X_train_scaled = preprocessor.transform(X_train_full)
X_val_scaled = preprocessor.transform(X_val_full)
X_test_scaled = preprocessor.transform(test_df[feature_cols])


## 6. 클래스 불균형 처리: SMOTE

핵심 문법:

- `SMOTE(k_neighbors=2)`
- `fit_resample(X, y)`

원리:

- 소수 클래스 주변의 이웃을 참고해서 새로운 가상 샘플을 만든다.
- 단순 복제가 아니라, 기존 샘플 사이를 보간해서 데이터를 늘린다.

중요 포인트:

- SMOTE는 반드시 `Train`에만 적용한다.
- `Val/Test`에 적용하면 현실 분포가 깨져서 평가가 왜곡된다.


In [ ]:
smote = SMOTE(k_neighbors=2)
X_train_resampled, y_train_resampled = smote.fit_resample(
    X_train_scaled, y_train
)


## 7. kNN 학습과 튜닝

핵심 문법:

- `KNeighborsClassifier(n_neighbors=k)`
- `.fit(X, y)`
- `.predict(X)`
- `accuracy_score(y_true, y_pred)`
- `f1_score(..., average='macro')`

원리:

- kNN은 가장 가까운 이웃 `k`개의 라벨을 보고 다수결로 분류한다.
- `k`가 너무 작으면 노이즈에 민감하고, 너무 크면 경계가 둔해질 수 있다.

평가 포인트:

- 클래스 불균형이 있으면 `accuracy`만 보면 착시가 생길 수 있다.
- 그래서 각 클래스를 균형 있게 보는 `macro F1`이 더 중요하다.


In [ ]:
best_k = 1
best_score = 0

for k in [1, 3, 5, 7, 9]:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_resampled, y_train_resampled)
    val_preds = knn.predict(X_val_scaled)

    acc = accuracy_score(y_val, val_preds)
    f1 = f1_score(y_val, val_preds, average='macro')

    if f1 > best_score:
        best_score = f1
        best_k = k


## 8. 분류 성적표와 혼동행렬

핵심 문법:

- `classification_report(...)`
- `ConfusionMatrixDisplay.from_predictions(...)`

원리:

- 단순 accuracy는 전체 정답률만 보여준다.
- 하지만 혼동행렬은 어떤 클래스를 어떤 클래스로 헷갈렸는지 보여준다.
- 즉, 모델의 약점을 분석하는 도구라고 보면 된다.


In [ ]:
final_knn_model = KNeighborsClassifier(n_neighbors=best_k)
final_knn_model.fit(X_train_resampled, y_train_resampled)

best_val_preds = final_knn_model.predict(X_val_scaled)

print(classification_report(y_val, best_val_preds, target_names=['Auto (0)', 'Manual (1)']))
ConfusionMatrixDisplay.from_predictions(y_val, best_val_preds)


## 9. PCA와 Decision Boundary

핵심 문법:

- `PCA(n_components=2)`
- `fit_transform(X)`
- `np.meshgrid(...)`
- `np.c_[a, b]`
- `ravel()`
- `inverse_transform(X_2d)`
- `reshape(shape)`

원리:

- 고차원 데이터를 PCA로 2차원에 투영해 시각화한다.
- 2차원 평면 위에 격자를 만든 뒤, 그 점들을 다시 원래 차원으로 복원해서 모델에 넣는다.
- 이렇게 하면 실제 고차원 모델의 결정 경계를 2차원 단면처럼 볼 수 있다.


In [ ]:
pca_model = PCA(n_components=2)
X_2d = pca_model.fit_transform(X_train_resampled)

xx, yy = np.meshgrid(
    np.arange(X_2d[:, 0].min() - 1, X_2d[:, 0].max() + 1, 0.05),
    np.arange(X_2d[:, 1].min() - 1, X_2d[:, 1].max() + 1, 0.05),
)
mesh_2d = np.c_[xx.ravel(), yy.ravel()]
mesh_high_dim = pca_model.inverse_transform(mesh_2d)
Z = final_knn_model.predict(mesh_high_dim).reshape(xx.shape)


## 10. 최종 제출 파일 생성

핵심 문법:

- `.copy()`
- `pd.DataFrame({...})`
- `.to_csv(filename, index=False)`

원리:

- 검증 단계에서 최적 `k`를 찾은 뒤, 과거 데이터 전체(`trainval`)로 다시 학습한다.
- 그다음 미래 데이터(`test`)를 예측해서 제출 파일을 만든다.

흐름:

- `Train/Val` 분리는 튜닝용
- 최종 제출은 `trainval` 전체 재학습


In [ ]:
X_full = trainval_df[feature_cols].copy()
y_full = trainval_df['target']

preprocessor.fit(X_full)
X_full_scaled = preprocessor.transform(X_full)
X_test_scaled = preprocessor.transform(test_df[feature_cols])

X_full_resampled, y_full_resampled = smote.fit_resample(X_full_scaled, y_full)

final_knn_model = KNeighborsClassifier(n_neighbors=best_k)
final_knn_model.fit(X_full_resampled, y_full_resampled)

test_preds = final_knn_model.predict(X_test_scaled)
submission_df = pd.DataFrame({'vehicle_id': test_df['vehicle_id'], 'target': test_preds})
submission_df.to_csv('submission_홍길동.csv', index=False)


## 11. NumPy로 kNN 직접 구현

이 부분은 함수 문법과 클래스 흐름을 같이 보는 것이 중요하다.

### `__init__`

- 객체가 생성될 때 자동 호출되는 초기화 메서드다.
- 여기서 `pass`를 쓴 것은 아직 초기화할 내용을 따로 넣지 않았기 때문이다.
- 즉, 생성자는 열어 두었지만 지금은 아무 동작도 하지 않는 상태다.

### `fit`

- kNN은 가중치를 학습하는 모델이 아니라, 훈련 데이터를 저장해 두는 모델이다.
- 그래서 `fit`의 핵심은 `self.X_train`, `self.y_train`에 데이터를 저장하는 것이다.

### `compute_distances`

핵심 식:

`||a-b||^2 = ||a||^2 - 2a·b + ||b||^2`

- 이 식을 쓰면 이중 반복문 없이 테스트 샘플과 학습 샘플 사이의 거리를 한 번에 계산할 수 있다.
- `axis=1`, `keepdims=True`, 전치(`.T`)가 왜 필요한지 같이 보는 것이 중요하다.

### `predict`

- 거리 행렬에서 가까운 `k`개의 인덱스를 찾는다.
- 그 인덱스의 라벨을 모아 `np.bincount()`로 개수를 세고 `np.argmax()`로 다수 클래스를 고른다.


In [ ]:
class KNearestNeighbor_Numpy:
    def __init__(self):
        pass

    def fit(self, X, y):
        self.X_train = np.array(X)
        self.y_train = np.array(y)

    def compute_distances(self, X_test):
        X_test = np.array(X_test)

        test_sq = np.sum(np.square(X_test), axis=1, keepdims=True)
        train_sq = np.sum(np.square(self.X_train), axis=1)
        cross_term = np.dot(X_test, self.X_train.T)

        dists = np.sqrt(np.maximum(test_sq - 2 * cross_term + train_sq, 0.0))
        return dists

    def predict(self, X_test, k=1):
        dists = self.compute_distances(X_test)
        num_test = dists.shape[0]
        y_pred = np.zeros(num_test)

        for i in range(num_test):
            closest_idx = np.argsort(dists[i, :])[:k]
            closest_y = self.y_train[closest_idx]
            counts = np.bincount(closest_y.astype(int))
            y_pred[i] = np.argmax(counts)

        return y_pred


## 12. 외우면 좋은 핵심 문법

```python
df['new_col'] = df['old_col'].apply(lambda x: ...)
pd.to_numeric(series, errors='coerce')
series.astype(str)
train_test_split(X, y, test_size=0.2)
Pipeline(steps=[...])
ColumnTransformer(transformers=[...])
preprocessor.fit(X_train)
preprocessor.transform(X_val)
smote.fit_resample(X, y)
model.fit(X, y)
model.predict(X)
f1_score(y_true, y_pred, average='macro')
PCA(n_components=2)
pca.fit_transform(X)
pca.inverse_transform(X_2d)
np.meshgrid(...)
np.c_[a, b]
np.argsort(arr)[:k]
np.bincount(labels.astype(int))
submission_df.to_csv('file.csv', index=False)
```
